In [4]:
import pymilvus

In [5]:
pymilvus.connections.connect(host='localhost', port='19530')

In [6]:
from pymilvus import utility

In [6]:
# A resource group name should be a string of 1 to 255 characters, starting with a letter or an underscore (_) and containing only numbers, letters, and underscores (_).
name = "rg"
node_num = 0

# create a resource group that exactly hold no query node.
try:
    utility.create_resource_group(name, config=utility.ResourceGroupConfig(
        requests={"node_num": node_num},
        limits={"node_num": node_num},
    ), using='default')
    print(f"Succeeded in creating resource group {name}.")
except Exception:
    print("Failed to create the resource group.")

Succeeded in creating resource group rg.


In [8]:
rgs = utility.list_resource_groups(using='default')
print(f"Resource group list: {rgs}")

Resource group list: ['__default_resource_group', '__pending_nodes', 'rg1', 'rg2']


In [16]:
info = utility.describe_resource_group("rg2", using="default")
print(f"Resource group description: {info}")

Resource group description: ResourceGroupInfo:
<name:rg2>,
<capacity:1>,
<num_available_node:0>,
<num_loaded_replica:{}>,
<num_outgoing_node:{}>,
<num_incoming_node:{}>,
<config:requests {
  node_num: 1
}
limits {
  node_num: 1
}
transfer_from {
  resource_group: "__pending_nodes"
}
transfer_to {
  resource_group: "__pending_nodes"
}
>,
<nodes:[]>


In [12]:
source = '__default_resource_group'
target = 'rg'
expected_num_nodes_in_default = 0
expected_num_nodes_in_rg = 1

try:
    utility.update_resource_groups({
        source: utility.ResourceGroupConfig(
            requests={"node_num": expected_num_nodes_in_default},
            limits={"node_num": expected_num_nodes_in_default},
        ),
        target: utility.ResourceGroupConfig(
            requests={"node_num": expected_num_nodes_in_rg},
            limits={"node_num": expected_num_nodes_in_rg},
        )
    }, using="default")
    print(f"Succeeded in moving 1 node(s) from {source} to {target}.")
except Exception:
    print("Something went wrong while moving nodes.")

Succeeded in moving 1 node(s) from __default_resource_group to rg.


In [14]:
try:
    utility.update_resource_groups({
        "rg": utility.ResourceGroupConfig(
            requests={"node_num": 0},
            limits={"node_num": 0},
        ),
    }, using="default")
    utility.drop_resource_group("rg", using="default")
    print(f"Succeeded in dropping {source}.")
except Exception:
    print(f"Something went wrong while dropping {source}.")

Succeeded in dropping __default_resource_group.


In [7]:
from pymilvus.client.types import ResourceGroupConfig


_PENDING_NODES_RESOURCE_GROUP="__pending_nodes"

def init_cluster(node_num: int):
    print(f"Init cluster with {node_num} nodes, all nodes will be put in default resource group")
    # create a pending resource group, which can used to hold the pending nodes that do not hold any data.
    utility.create_resource_group(name=_PENDING_NODES_RESOURCE_GROUP, config=ResourceGroupConfig(
        requests={"node_num": 0}, # this resource group can hold 0 nodes, no data will be load on it.
        limits={"node_num": 10000}, # this resource group can hold at most 10000 nodes 
    ))

    # create a default resource group, which can used to hold the nodes that all initial node in it.
    utility.update_resource_groups({
        "__default_resource_group": ResourceGroupConfig(
            requests={"node_num": node_num},
            limits={"node_num": node_num},
            transfer_from=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}], # recover missing node from pending resource group at high priority.
            transfer_to=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}], # recover redundant node to pending resource group at low priority.
        )})
    utility.create_resource_group(name="rg1", config=ResourceGroupConfig(
        requests={"node_num": 0},
        limits={"node_num": 0},
        transfer_from=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}], 
        transfer_to=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
    ))
    utility.create_resource_group(name="rg2", config=ResourceGroupConfig(
        requests={"node_num": 0},
        limits={"node_num": 0},
        transfer_from=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}], 
        transfer_to=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
    ))

init_cluster(1)

Init cluster with 1 nodes, all nodes will be put in default resource group


In [13]:
utility.update_resource_groups({
    "rg1": ResourceGroupConfig(
        requests={"node_num": 3},
        limits={"node_num": 3},
        transfer_from=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
        transfer_to=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
    ),
    "rg2": ResourceGroupConfig(
        requests={"node_num": 1},
        limits={"node_num": 1},
        transfer_from=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
        transfer_to=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
    ),
})

In [17]:
utility.update_resource_groups({
    "rg1": ResourceGroupConfig(
        requests={"node_num": 3},
        limits={"node_num": 3},
        transfer_from=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
        transfer_to=[{"resource_group": _PENDING_NODES_RESOURCE_GROUP}],
    ),
})